# ⚽ Partido: Equipo AZUL vs Equipo ROJO (inferencia)

Carga dos archivos `equipo_*.pt` entrenados por estudiantes distintos y los enfrenta en un partido 2 vs 2.
**Gana el primer equipo que llegue a 5 goles.** El resultado se muestra en un video con marcador.

Los agentes juegan en modo determinista (cada uno elige la acción más probable de su actor). El equipo rojo usa la misma red que fue entrenada como azul: VMAS espeja observaciones y acciones del rojo, por lo que es un duelo justo.

> No necesita GPU. Ejecuta las celdas en orden.

In [ ]:
!pip install -q vmas==1.5.2 benchmarl==1.5.2 imageio imageio-ffmpeg pyvirtualdisplay
!apt-get -qq install -y xvfb x11-utils python3-opengl libglu1-mesa > /dev/null 2>&1

In [ ]:
# Pantalla virtual para que VMAS pueda dibujar en Colab (si falla, se usa un dibujo alternativo con matplotlib)
try:
    from pyvirtualdisplay import Display
    _display = Display(visible=0, size=(1280, 800)); _display.start()
    print("Pantalla virtual lista")
except Exception as e:
    print("Sin pantalla virtual, se usará el render alternativo:", e)

## 1. Sube los dos equipos

In [ ]:
from google.colab import files
print("Sube el archivo del equipo AZUL (.pt)")
_up = files.upload(); ARCH_AZUL = list(_up.keys())[0]
print("Sube el archivo del equipo ROJO (.pt)")
_up = files.upload(); ARCH_ROJO = list(_up.keys())[0]
print(ARCH_AZUL, "vs", ARCH_ROJO)

In [ ]:
import io, base64, numpy as np, torch, vmas, imageio
from PIL import Image, ImageDraw, ImageFont
from tensordict import TensorDict
from torchrl.envs.utils import set_exploration_type, ExplorationType
from benchmarl.algorithms import MappoConfig
from benchmarl.environments import VmasTask
from benchmarl.experiment import Experiment, ExperimentConfig
from benchmarl.models.mlp import MlpConfig

# ---- Debe coincidir con la configuración FIJA del cuaderno de entrenamiento ----
TASK_FIJO = dict(
    max_steps=500,
    n_blue_agents=2, n_red_agents=2,
    ai_red_agents=True, ai_blue_agents=False,
    enable_shooting=False,
    observe_teammates=True, observe_adversaries=True, dict_obs=False,
)

def cargar_equipo(ruta):
    ck = torch.load(ruta, map_location="cpu")
    assert ck.get("formato") == "mappo-football-2v2-v1", f"{ruta}: formato desconocido"
    task = VmasTask.FOOTBALL.get_from_yaml(); task.config.update(TASK_FIJO)
    cfg = ExperimentConfig.get_from_yaml()
    cfg.sampling_device = cfg.train_device = cfg.buffer_device = "cpu"
    cfg.share_policy_params = True
    cfg.prefer_continuous_actions = False
    cfg.max_n_frames = 100; cfg.on_policy_collected_frames_per_batch = 100
    cfg.on_policy_n_envs_per_worker = 1; cfg.on_policy_minibatch_size = 100; cfg.on_policy_n_minibatch_iters = 1
    cfg.evaluation = False; cfg.render = False; cfg.loggers = []; cfg.create_json = False
    cfg.checkpoint_interval = 0; cfg.save_folder = "/content/tmp_exp"
    import os; os.makedirs(cfg.save_folder, exist_ok=True)
    exp = Experiment(task=task, algorithm_config=MappoConfig.get_from_yaml(),
                     model_config=MlpConfig.get_from_yaml(), critic_model_config=MlpConfig.get_from_yaml(),
                     seed=0, config=cfg)
    sd = exp.policy.state_dict()                      # incluye entradas auxiliares (__batch_size)
    esperadas = [k for k, v in sd.items() if torch.is_tensor(v)]
    faltan = set(esperadas) - set(ck["policy_state_dict"].keys())
    assert not faltan, f"Faltan pesos en {ruta}: {faltan}"
    for k in esperadas:
        sd[k] = ck["policy_state_dict"][k]            # sobrescribo solo los pesos entrenados
    exp.policy.load_state_dict(sd)
    sd_nuevo = exp.policy.state_dict()
    assert all(torch.equal(sd_nuevo[k], ck["policy_state_dict"][k]) for k in esperadas), "Los pesos no se cargaron"
    print(f"✔ {ck['equipo']} ({ck['integrantes']}) — {ck['frames_entrenados']:,} frames — vmas {ck['versiones']['vmas']}")
    return dict(nombre=ck["equipo"], policy=exp.policy, grupo=ck["grupo"], obs_dim=ck["obs_dim"])

AZUL = cargar_equipo(ARCH_AZUL)
ROJO = cargar_equipo(ARCH_ROJO)

## 2. Motor del partido

In [ ]:
GOLES_PARA_GANAR = 5
VIEWER = (720, 480)
_render_vmas_ok = True

def crear_env(semilla=0):
    kw = dict(TASK_FIJO, ai_red_agents=False, ai_blue_agents=False)   # los 4 jugadores los controlan las redes
    env = vmas.make_env(scenario="football", num_envs=1, device="cpu", continuous_actions=False,
                        seed=semilla, viewer_size=VIEWER, **kw)
    nombres = [a.name for a in env.agents]
    assert nombres == ["agent_blue_0", "agent_blue_1", "agent_red_0", "agent_red_1"], nombres
    return env

def elegir_acciones(equipo, obs_list):
    obs = torch.stack(obs_list, dim=1)
    assert obs.shape[-1] == equipo["obs_dim"], "obs_dim distinto: el equipo se entrenó con otra configuración"
    n, na = obs.shape[0], obs.shape[1]
    g = equipo["grupo"]
    td = TensorDict({g: TensorDict({"observation": obs}, batch_size=[n, na])}, batch_size=[n])
    with torch.no_grad(), set_exploration_type(ExplorationType.DETERMINISTIC):
        td = equipo["policy"](td)
    return td.get((g, "action")).reshape(n, na)

# ---------- dibujo ----------
def _frame_matplotlib(env):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Circle, Rectangle
    s = env.scenario; Lp, W, gs = s.pitch_length, s.pitch_width, s.goal_size
    fig = plt.figure(figsize=(VIEWER[0]/100, VIEWER[1]/100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(-Lp/2-0.2, Lp/2+0.2); ax.set_ylim(-W/2-0.1, W/2+0.1); ax.axis("off")
    ax.add_patch(Rectangle((-Lp/2-0.2, -W/2-0.1), Lp+0.4, W+0.2, color="#2e7d32"))
    ax.add_patch(Rectangle((-Lp/2, -W/2), Lp, W, fill=False, ec="white", lw=2))
    ax.plot([0, 0], [-W/2, W/2], color="white", lw=1.5)
    for sx in (-1, 1): ax.add_patch(Rectangle((sx*Lp/2 - (0.1 if sx < 0 else 0), -gs/2), 0.1, gs, fill=False, ec="white", lw=2))
    for a in s.blue_agents: ax.add_patch(Circle(a.state.pos[0].tolist(), s.agent_size*1.6, color="#377eb8"))
    for a in s.red_agents:  ax.add_patch(Circle(a.state.pos[0].tolist(), s.agent_size*1.6, color="#e41a1c"))
    ax.add_patch(Circle(s.ball.state.pos[0].tolist(), s.ball_size*1.6, color="white", ec="black"))
    fig.canvas.draw(); img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy(); plt.close(fig); return img

def dibujar(env, marcador, nombre_azul, nombre_rojo, texto=None):
    global _render_vmas_ok
    frame = None
    if _render_vmas_ok:
        try:
            frame = env.render(mode="rgb_array", env_index=0)
        except Exception as e:
            print("⚠ Render de VMAS no disponible, uso matplotlib:", type(e).__name__); _render_vmas_ok = False
    if frame is None: frame = _frame_matplotlib(env)
    img = Image.fromarray(np.asarray(frame).astype(np.uint8)); d = ImageDraw.Draw(img)
    try: f = ImageFont.load_default(size=22); fg = ImageFont.load_default(size=56)
    except Exception: f = fg = ImageFont.load_default()
    d.rectangle([0, 0, img.width, 34], fill=(0, 0, 0))
    d.text((10, 5), f"{nombre_azul} (azul)  {marcador['azul']}  -  {marcador['rojo']}  {nombre_rojo} (rojo)", fill=(255, 255, 255), font=f)
    if texto: d.text((img.width//2 - 80, img.height//2 - 30), texto, fill=(255, 235, 59), font=fg)
    w, h = img.width - img.width % 2, img.height - img.height % 2
    return np.asarray(img.crop((0, 0, w, h)))

def jugar_partido(azul, rojo, goles=GOLES_PARA_GANAR, max_pasos=8000, video=None, cada=2, semilla=0):
    env = crear_env(semilla); Lp = env.scenario.pitch_length
    writer = imageio.get_writer(video, fps=20, macro_block_size=1) if video else None
    obs = env.reset(); marcador = {"azul": 0, "rojo": 0}; paso = 0
    while max(marcador.values()) < goles and paso < max_pasos:
        aa, ar = elegir_acciones(azul, obs[:2]), elegir_acciones(rojo, obs[2:])
        acc = [aa[:, i:i+1] for i in range(2)] + [ar[:, i:i+1] for i in range(2)]
        obs, _, done, _ = env.step(acc); paso += 1
        gol = None
        if done.any():
            x = env.scenario.ball.state.pos[0, 0].item()
            if x > Lp/2: marcador["azul"] += 1; gol = "azul"
            elif x < -Lp/2: marcador["rojo"] += 1; gol = "rojo"
        if writer and (paso % cada == 0 or gol):
            fr = dibujar(env, marcador, azul["nombre"], rojo["nombre"], texto=f"¡GOL {gol.upper()}!" if gol else None)
            for _ in range(15 if gol else 1): writer.append_data(fr)
        if done.any(): obs = env.reset_at(0)
    if marcador["azul"] > marcador["rojo"]: ganador = azul["nombre"]
    elif marcador["rojo"] > marcador["azul"]: ganador = rojo["nombre"]
    else: ganador = None
    if writer:
        fr = dibujar(env, marcador, azul["nombre"], rojo["nombre"], texto="FIN")
        for _ in range(40): writer.append_data(fr)
        writer.close()
    return dict(marcador=marcador, ganador=ganador, pasos=paso)

def mostrar_video(ruta, ancho=800):
    from IPython.display import HTML
    b64 = base64.b64encode(open(ruta, "rb").read()).decode()
    return HTML(f'<video width="{ancho}" controls autoplay loop><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

## 3. ▶️ ¡Jugar el partido!
Se simula el partido completo (offline) y luego se reproduce el video.

In [ ]:
res = jugar_partido(AZUL, ROJO, video="partido.mp4", semilla=0)
print("Marcador final:", AZUL["nombre"], res["marcador"]["azul"], "-", res["marcador"]["rojo"], ROJO["nombre"])
print("🏆 Ganador:", res["ganador"] if res["ganador"] else "nadie llegó a 5 goles (empate al límite de pasos)")
mostrar_video("partido.mp4")

## 4. (Opcional) Serie de partidos sin video
Un solo partido tiene mucho azar. Esta celda juega varios partidos rápidos **intercambiando de lado** y cuenta victorias.

In [ ]:
def serie(azul, rojo, n=6, max_pasos=4000):
    vic = {azul["nombre"]: 0, rojo["nombre"]: 0}; goles = {azul["nombre"]: 0, rojo["nombre"]: 0}; empates = 0
    for i in range(n):
        a, r = (azul, rojo) if i % 2 == 0 else (rojo, azul)   # intercambio de lados
        res = jugar_partido(a, r, max_pasos=max_pasos, semilla=100 + i)
        goles[a["nombre"]] += res["marcador"]["azul"]; goles[r["nombre"]] += res["marcador"]["rojo"]
        if res["ganador"]: vic[res["ganador"]] += 1
        else: empates += 1
        print(f"Partido {i+1}: {a['nombre']} {res['marcador']['azul']} - {res['marcador']['rojo']} {r['nombre']}")
    print("\nVictorias:", vic, "| Empates:", empates, "| Goles:", goles)

serie(AZUL, ROJO, n=6)